In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_025_02_12_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_018_01_30_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_011_02_30_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_026_02_48_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_004_02_6_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_002_02_36_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_025_01_36_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_017_02_36_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_018_02_60_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_015_02_54_image.png
/kaggle/input/emergency-sign-language/Image_Data/call/colored_call_006_02_66_image.png
/kaggle/input/emergency-sign-language/Image_

In [2]:
!pip install -q mediapipe opencv-python-headless albumentations tensorflow-addons
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import mediapipe as mp
import albumentations as A
from pathlib import Path
import os
print("TensorFlow:", tf.__version__)


ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons


2025-12-29 14:09:49.350019: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767017389.537325      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767017389.589307      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767017390.030063      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767017390.030098      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767017390.030101      24 computation_placer.cc:177] computation placer alr

ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
DATA_DIR = Path('/kaggle/input/emergency-sign-language')
classes = sorted([d.name for d in DATA_DIR.iterdir()])
print("Classes:", classes)
print("Samples per class:")
for cls in classes:
    print(f"{cls}: {len(list((DATA_DIR/cls).iterdir()))}")
    
# Visualize samples
fig, axs = plt.subplots(2, 4, figsize=(16,8))
for i, cls in enumerate(classes):
    img_path = next((DATA_DIR/cls).iterdir())
    img = plt.imread(img_path)
    axs[i//4, i%4].imshow(img)
    axs[i//4, i%4].set_title(cls)
    axs[i//4, i%4].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

# Collect ALL image paths and labels from raw dataset
all_paths = []
all_labels = []
classes = sorted([d.name for d in Path('/kaggle/input/emergency-sign-language/Image_Data').iterdir()])
class_to_idx = {cls: i for i, cls in enumerate(classes)}

print("Classes found:", classes)

for i, cls in enumerate(classes):
    cls_path = Path('/kaggle/input/emergency-sign-language/Image_Data') / cls
    cls_files = list(cls_path.glob('*.jpg')) + list(cls_path.glob('*.jpeg')) + list(cls_path.glob('*.png'))
    all_paths.extend([str(p) for p in cls_files])
    all_labels.extend([i] * len(cls_files))
    print(f"{cls}: {len(cls_files)} images")

print(f"\nTotal images: {len(all_paths)}")

# Split: 80/10/10 stratified
X_temp, X_val, y_temp, y_val = train_test_split(
    all_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X_temp, y_temp, test_size=0.125, stratify=y_temp, random_state=42  # 0.125 of total = 10%
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

def load_and_preprocess_image(img_path, label):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

# Create datasets with augmentation for training only
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

print("Datasets created successfully!")
# Add this augmentation to FIXED Cell 3 after train_ds line:
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    zoom_range=0.2
)

# Replace train_ds creation with:
def augment_image(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    return img, label

train_ds = train_ds.map(augment_image, num_parallel_calls=AUTOTUNE)



In [ ]:
def create_simple_cnn(num_classes=len(classes)):
    model = keras.Sequential([
        layers.Conv2D(32, 3, activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = create_simple_cnn()
model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(factor=0.2, patience=5, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint('/kaggle/working/best_model.h5', save_best_only=True, monitor='val_accuracy')
]

history = model.fit(
    train_ds, 
    epochs=100,  # More epochs
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# Plot training curves
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Model Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.legend()
plt.show()

# Test set evaluation
test_loss, test_acc = model.evaluate(test_ds)
print(f"✅ FINAL TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.1f}%)")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Predictions on test set
y_test_pred = model.predict(test_ds)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)

# Get test labels
y_test_actual = np.concatenate([y for x, y in test_ds], axis=0)

print("📊 CLASSIFICATION REPORT:")
print(classification_report(y_test_actual, y_test_pred_classes, target_names=classes))

# Confusion Matrix
cm = confusion_matrix(y_test_actual, y_test_pred_classes)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Save final models
model.save('/kaggle/working/emergency_sign_model.keras')
model.save('/kaggle/working/emergency_sign_model.h5')

# Create submission-ready zip
!zip -r /kaggle/working/emergency_sign_model.zip /kaggle/working/*.h5 /kaggle/working/*.keras

print("🎉 MODEL SAVED!")
print("✅ Download from: /kaggle/working/")
print("- emergency_sign_model.keras (Keras format)")
print("- emergency_sign_model.h5 (TensorFlow format)")
print("- emergency_sign_model.zip (all files)")
print(f"\n🏆 FINAL PERFORMANCE: {test_acc*100:.1f}% accuracy")


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image
import mediapipe as mp

# Load your trained model (download from Kaggle)
model = tf.keras.models.load_model('emergency_sign_model.keras')
classes = ['accident', 'call', 'doctor', 'help', 'hot', 'lose', 'pain', 'thief']
IMG_SIZE = 224

# MediaPipe hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)
    
    # Predict gesture
    frame_resized = cv2.resize(frame_rgb, (IMG_SIZE, IMG_SIZE))
    frame_norm = np.expand_dims(frame_resized.astype(np.float32)/255.0, 0)
    pred = model.predict(frame_norm, verbose=0)
    gesture = classes[np.argmax(pred)]
    confidence = np.max(pred) * 100
    
    # Draw hand landmarks
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
    
    # Display prediction
    cv2.putText(frame, f'{gesture}: {confidence:.1f}%', (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    
    # EMERGENCY ALERT for "help" or "accident"
    if gesture in ['help', 'accident'] and confidence > 80:
        cv2.putText(frame, '🚨 EMERGENCY ALERT! 🚨', (10, 70), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
    
    cv2.imshow('Emergency Sign Language Detection', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
requirements = """tensorflow
opencv-python
mediapipe
pillow
numpy
"""
with open('/kaggle/working/requirements.txt', 'w') as f:
    f.write(requirements)
print("✅ Download requirements.txt too!")
